# Task 1: Data Cleaning and Preprocessing
## Dataset: Iris Dataset

**Objective:** Load the dataset, identify and handle missing values, remove duplicate rows, and standardize inconsistent data formats.

**Contents:**
1. Load the Dataset
2. Initial Data Inspection
3. Identify and Handle Missing Values
4. Remove Duplicate Rows
5. Standardize Data Formats
6. Additional Data Quality Checks
7. Save the Cleaned Dataset
8. Before vs. After Summary


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load the Dataset

In [2]:
df = pd.read_csv('iris.csv')
print("Shape:", df.shape)
df.head()

Shape: (150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## 2. Initial Data Inspection

Check data types, column names, and get a general sense of data quality before cleaning.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


In [4]:
print("Columns:", list(df.columns))
print()
print("Unique species values:", df['species'].unique())
print("Species value counts:")
print(df['species'].value_counts())

Columns: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

Unique species values: <StringArray>
['setosa', 'versicolor', 'virginica']
Length: 3, dtype: str
Species value counts:
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


**Observation:** The dataset has 4 numerical columns (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`) and one categorical column (`species`) with 3 classes. Column names are already lowercase and consistent (no spaces or mixed casing to fix).

## 3. Identify and Handle Missing Values

In [5]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
sepal_length,0,0.0
sepal_width,0,0.0
petal_length,0,0.0
petal_width,0,0.0
species,0,0.0


**Observation:** There are **no missing values** anywhere in the dataset. In a case like this, no imputation or removal is required — but the check itself is still an essential first step of any cleaning pipeline, since missing values can appear silently (e.g., as blanks, `NaN`, or placeholder strings) and should never be assumed absent without checking.

*(For documentation purposes: if missing values had been found in the numeric columns, the standard approach would be **imputation with the median per species** — since sepal/petal measurements differ meaningfully by species, a single global mean would distort the data. Rows missing the `species` label itself would typically be dropped, since it can't be reasonably imputed.)*

In [6]:
df_clean = df.copy()

## 4. Remove Duplicate Rows

Check for fully duplicated rows.

In [7]:
full_dupes = df_clean.duplicated().sum()
print(f"Fully duplicated rows: {full_dupes}")
df_clean[df_clean.duplicated(keep=False)].sort_values(list(df_clean.columns))

Fully duplicated rows: 3


,sepal_length,sepal_width,petal_length,petal_width,species
9,4.9,3.1,1.5,0.1,setosa
34,4.9,3.1,1.5,0.1,setosa
37,4.9,3.1,1.5,0.1,setosa
101,5.8,2.7,5.1,1.9,virginica
142,5.8,2.7,5.1,1.9,virginica


In [8]:
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Shape before: {df.shape}")
print(f"Shape after removing duplicates: {df_clean.shape}")

Shape before: (150, 5)
Shape after removing duplicates: (147, 5)


**Observation:** The dataset contained **3 duplicate rows** (2 extra copies of one `setosa` row, and 1 extra copy of a `virginica` row) — these were removed, keeping only the first occurrence of each.

## 5. Standardize Data Formats

Check for inconsistent formatting in the categorical column (`species`) — stray whitespace, inconsistent casing, or typos — and confirm the numeric columns have consistent types.

In [9]:
before = df_clean['species'].copy()
df_clean['species'] = df_clean['species'].str.strip().str.lower()

changed = (before != df_clean['species']).sum()
print(f"Species values changed by standardization: {changed}")
print("Unique species after standardization:", sorted(df_clean['species'].unique()))

Species values changed by standardization: 0
Unique species after standardization: ['setosa', 'versicolor', 'virginica']


In [10]:
print("Data types of numeric columns:")
print(df_clean[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].dtypes)

Data types of numeric columns:
sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
dtype: object


**Observation:** The `species` column was already clean (lowercase, no stray whitespace, only 3 valid categories) — standardization is applied anyway as a safeguard. All 4 measurement columns are consistently stored as floats, so no type conversion was needed.

## 6. Additional Data Quality Checks

Sanity checks appropriate for this dataset: negative measurements (physically impossible for sepal/petal dimensions) and implausible outlier values.

In [11]:
measurement_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

for c in measurement_cols:
    neg = (df_clean[c] < 0).sum()
    zero = (df_clean[c] == 0).sum()
    print(f"{c:15s} -> negative: {neg}, zero: {zero}")

print()
df_clean[measurement_cols].describe()

sepal_length    -> negative: 0, zero: 0
sepal_width     -> negative: 0, zero: 0
petal_length    -> negative: 0, zero: 0
petal_width     -> negative: 0, zero: 0



,sepal_length,sepal_width,petal_length,petal_width
count,147.000000,147.000000,147.000000,147.000000
mean,5.856463,3.055782,3.780272,1.208844
std,0.829100,0.437009,1.759111,0.757874
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.400000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


**Observation:** No negative or zero measurements were found — all values fall within a plausible biological range for iris flowers (a few centimeters), confirming there's no obvious data corruption.

## 7. Save the Cleaned Dataset

In [12]:
df_clean.to_csv('iris_cleaned.csv', index=False)
print("Saved cleaned dataset:", df_clean.shape)
df_clean.head()

Saved cleaned dataset: (147, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## 8. Before vs. After Summary

In [13]:
summary = pd.DataFrame({
    'Metric': [
        'Total rows',
        'Missing values (any column)',
        'Duplicate rows',
        'Species formatting'
    ],
    'Before': [
        len(df),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        'unchecked'
    ],
    'After': [
        len(df_clean),
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
        'standardized (lowercase, trimmed)'
    ]
})
summary

,Metric,Before,After
0,Total rows,150,147
1,Missing values (any column),0,0
2,Duplicate rows,3,0
3,Species formatting,unchecked,"standardized (lowercase, trimmed)"


**Summary:**
1. **Missing values** — None were found; the dataset was already complete across all 5 columns.
2. **Duplicates** — 3 duplicate rows were identified and removed (150 rows -> 147 rows).
3. **Format standardization** — The `species` column was standardized (trimmed, lowercased) as a safeguard, though it was already consistent; numeric columns were confirmed to have consistent float types.
4. **Extra quality checks** — No negative or zero measurements, confirming no obvious data corruption.
5. The cleaned dataset (`iris_cleaned.csv`) is now ready for downstream analysis (e.g., an EDA task).
